# 🏥 Healthcare Chatbot with Guardrails

A **healthcare chatbot** designed with multiple layers of guardrails to make interactions safer, protect sensitive patient information, and ensure that high-impact actions require human oversight.

The chatbot combines **input filtering, PII protection, human-in-the-loop approval, and output validation** to create a safer healthcare-oriented AI assistant.

### Key Capabilities

1. **🚫 Blocks Harmful or Off-Topic Requests**  
   Prevents the chatbot from processing inappropriate, harmful, or unrelated requests.

2. **🔐 Protects Patient PII**  
   Detects and protects sensitive information such as **email addresses and credit card numbers** before the agent processes the request.

3. **👨‍⚕️ Human Approval for Appointments**  
   Requires a human to approve appointment-booking actions before they are executed.

4. **⚕️ Validates Medical Outputs**  
   Checks the chatbot's responses and ensures that appropriate medical disclaimers are included.

### Why Guardrails Matter

Healthcare applications handle **sensitive information and high-impact decisions**, so simply having an LLM answer questions is not enough. Guardrails provide additional layers of protection around the agent, helping control what enters the system, what actions the agent can perform, and what information is returned to the user.

## Implementation of Chatbot

### Import Required Libraries

In [1]:
import os
from typing import Any

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    PIIMiddleware,
    HumanInTheLoopMiddleware,
    hook_config,
)
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

### Environment Setup and Groq Model

In [2]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("✅ Groq model initialized!")

✅ Groq model initialized!


### Healthcare Safety Filter

In [3]:
class HealthcareSafetyFilter(AgentMiddleware):
    """
    Block non-medical or harmful requests
    in a healthcare context.
    """

    BLOCKED_TOPICS = [
        "drug synthesis",
        "self-harm",
        "suicide method",
        "weapon",
        "hack"
    ]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        first_msg = state["messages"][0]

        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()

        for topic in self.BLOCKED_TOPICS:

            if topic in content:

                print(
                    f"🚫 Blocked — topic detected: '{topic}'"
                )

                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only "
                            "help with medical questions, appointments, "
                            "and health information. "
                            "If you're in crisis, please call 112 "
                            "or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }

        return None

### Medical Output Validator

In [4]:
class MedicalOutputValidator(AgentMiddleware):
    """
    Ensure healthcare responses include
    an appropriate medical disclaimer.
    """

    DISCLAIMER = (
        "\n\n⚕️ *This is general health information, "
        "not medical advice. Please consult a qualified "
        "healthcare professional.*"
    )

    @hook_config()
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():

            last_message.content += self.DISCLAIMER

        return None

### Healthcare Tools

In [5]:
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""

    return (
        f"Symptom information for: {symptoms}. "
        "Please consult a doctor for diagnosis."
    )


@tool
def book_appointment(
    patient_name: str,
    date: str,
    doctor: str
) -> str:
    """Book a medical appointment."""

    return (
        f"Appointment booked for {patient_name} "
        f"with Dr. {doctor} on {date}"
    )


@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""

    return (
        f"General information about {medication}. "
        "Always follow your doctor's prescription."
    )


print("✅ Healthcare tools created!")

✅ Healthcare tools created!


### Build Healthcare Chatbot

In [6]:
healthcare_bot = create_agent(
    model=model,

    tools=[
        search_symptoms,
        book_appointment,
        get_medication_info
    ],

    middleware=[

        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient email
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),

        # Guardrail 3: Mask credit card numbers
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),

        # Guardrail 4: Human approval for appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 5: Medical output validation
        MedicalOutputValidator(),
    ],

    # Required for HumanInTheLoopMiddleware
    checkpointer=InMemorySaver(),

    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, "
        "and help book appointments. "
        "Always be empathetic and remind users to consult "
        "a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


### Test 1: Safe Medical Query

In [7]:
config_t1 = {
    "configurable": {
        "thread_id": "healthcare_safe_001"
    }
}

result = healthcare_bot.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are symptoms of Type 2 Diabetes?"
            }
        ]
    },
    config=config_t1
)

print("=== Safe Medical Query ===")
print(result["messages"][-1].content)

=== Safe Medical Query ===
Type 2 diabetes often develops slowly, and many people don’t notice any symptoms at first. When symptoms do appear, they can include:

| Symptom | What to look for |
|---------|-------------------|
| **Increased thirst (polydipsia)** | Feeling unusually thirsty even after drinking fluids. |
| **Frequent urination (polyuria)** | Needing to go to the bathroom more often, especially at night. |
| **Unexplained weight loss** | Losing weight without trying, despite eating normally. |
| **Fatigue** | Persistent tiredness or low energy that doesn’t improve with rest. |
| **Blurred vision** | Vision that becomes fuzzy or changes temporarily. |
| **Slow‑healing cuts or infections** | Small wounds, sores, or infections that take longer to heal. |
| **Numbness or tingling in hands/feet** | A “pins‑and‑needles” feeling, often called peripheral neuropathy. |
| **Darkened skin patches** | Particularly around the neck or armpits (a condition called acanthosis nigricans). |


### Test 2: PII Redaction

In [8]:
config_t2 = {
    "configurable": {
        "thread_id": "healthcare_pii_001"
    }
}

result = healthcare_bot.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "My email is patient123@gmail.com. "
                    "What can I take for a headache?"
                )
            }
        ]
    },
    config=config_t2
)

print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
I’m sorry you’re dealing with a headache—those can be really frustrating.  

**Common over‑the‑counter options** that many people find helpful include:

| Medication | Typical dose for adults* | How it works | Things to watch for |
|------------|--------------------------|--------------|---------------------|
| **Acetaminophen (Tylenol)** | 500 mg–1 g every 4–6 hours (max 3 g per day) | Reduces pain and fever | Avoid if you have liver disease or drink alcohol heavily |
| **Ibuprofen (Advil, Motrin)** | 200 mg–400 mg every 6–8 hours (max 1.2 g per day OTC) | Anti‑inflammatory, pain reliever | Take with food; not for people with stomach ulcers, kidney problems, or certain heart conditions |
| **Naproxen (Aleve)** | 220 mg every 8–12 hours (max 660 mg per day) | Longer‑acting anti‑inflammatory | Same cautions as ibuprofen; avoid if you have severe heart disease |
| **Aspirin** | 325 mg–650 mg every 4–6 hours (max 4 g per day) | Anti‑inflammatory, pain reliever |

### Test 3: Blocked Request (Off-topic / Harmful Request)

In [9]:
config_t3 = {
    "configurable": {
        "thread_id": "healthcare_blocked_001"
    }
}

result = healthcare_bot.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How do I synthesize drugs at home?"
            }
        ]
    },
    config=config_t3
)

print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I’m sorry, but I can’t help with that. If you have any health concerns or questions about medications, please consult a qualified medical professional.

⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


### Test 4: Appointment Booking — Requires Human Approval

In [10]:
config_t4 = {
    "configurable": {
        "thread_id": "healthcare_appointment_001"
    }
}

result = healthcare_bot.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Book me an appointment with "
                    "Dr. Sharma on March 15"
                )
            }
        ]
    },
    config=config_t4
)

print("=== Appointment Booking — Awaiting Approval ===")
print(result)

=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='0b8a1372-d308-4ad0-8ebe-a22bba6bcdc0'), AIMessage(content='I’m happy to help set that up! Could you please let me know the name you’d like the appointment booked under? (If you have any preferred time of day, feel free to share that as well.)\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*', additional_kwargs={'reasoning_content': 'User wants to book appointment with Dr. Sharma on March 15. Need patient name. Not provided. Should ask for patient name. Also confirm date format? Probably need to ask.'}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 226, 'total_tokens': 316, 'completion_time': 0.190828715, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.012295979, 'prompt_tokens_details

### Approve Appointment
Use the same config_t4 because this is resuming the interrupted conversation.

In [11]:
approved = healthcare_bot.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config_t4
)

print("\n=== After Approval ===")
print(approved["messages"][-1].content)


=== After Approval ===
I’m happy to help set that up! Could you please let me know the name you’d like the appointment booked under? (If you have any preferred time of day, feel free to share that as well.)


---

## 🎯 Conclusion

This healthcare chatbot demonstrates how **multiple guardrails can work together** to make an AI agent safer and more controlled.

Throughout this implementation, we used:

- 🚫 **Input filtering** to block harmful or irrelevant requests
- 🔐 **PII protection** to safeguard sensitive patient information
- 👨‍⚕️ **Human-in-the-loop approval** for appointment booking
- ⚕️ **Output validation** to ensure appropriate healthcare responses
- 🧠 **LangGraph checkpointing** to maintain state during human approval workflows

The key takeaway is that **guardrails should be treated as multiple layers of protection rather than a single safety check**.

> **A reliable healthcare AI system should not only generate helpful responses, but also control what it receives, what actions it performs, and what it returns to the user.**

### 🚀 Final Takeaway

This example shows how LangChain middleware can be combined to build a more **safe, privacy-aware, and human-supervised AI agent** for real-world healthcare applications.

---